# Deep Learning Final Project  
**Course:** COMP 691 – Deep Learning  
**Instructor:** Prof. Kefaya QADDDOUM  
**Student:** Sam Collin  
**Student ID:** 40316218  
**Submission Date:** April 18, 2025  

---

### Notebook Overview

This notebook presents the **final version** of my deep learning project for Challenge 2.

It includes:
- The **best model configurations** for each of the two challenges selected after extensive experimentation (ResNet-18 and ResNet-50),
- Fully **reproducible training and evaluation code**,
- No dependency on Weights & Biases (W\&B) for logging or sweeping that can hardened the understanding if not familiar with.

To keep this notebook clean and lightweight, advanced code and exploratory tools (e.g., W\&B sweeps, full grid search, MAML implementation) are placed in the **code appendix**, which is available separately and referenced in the report.

---



# Challenge 1: Learning From Scratch (No External Help)

In this challenge, the model had to be trained from scratch using only the provided few-shot data (2 randomly selected classes, 25 images per class). No pre-trained models or external resources were allowed.

My approach was to build a lightweight convolutional neural network (CNN) tailored to learn effectively from very limited data. I focused on balancing model capacity and regularization to avoid overfitting.

All exploratory experiments—such as architecture variants, learning rate tuning, and layer configurations—are presented in the appendix code and described in the report. This notebook only includes the final version of the model that gave the most stable results without external help.


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Wider Network with Batch Normalization and Dropout
class Net_wide_BatchNorm_Dropout(nn.Module):
    def __init__(self, dropout_prob=0.5):
        super(Net_wide_BatchNorm_Dropout, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, stride=2)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, stride=2)
        self.bn4 = nn.BatchNorm2d(64)
        self.fc = nn.Linear(64*5*5, 10)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.dropout(x)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.dropout(x)
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [10]:
def train(model, device, train_loader, optimizer, epoch, display=True):
    model.train()
    running_loss = 0.
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Slight modif to the loss to have the mean over all the batches instead of the last one
    avg_loss = running_loss / len(train_loader)

    if display:
      print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
          epoch, batch_idx * len(data), len(train_loader.dataset),
          100. * batch_idx / len(train_loader), avg_loss))


def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.cross_entropy(output, target, size_average=False).item() # sum up batch loss
            pred = output.max(1, keepdim=True)[1] # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.2f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        accuracy))
    return accuracy

In [11]:
from numpy.random import RandomState
import numpy as np
import torch.optim as optim
from torch.utils.data import Subset

from torchvision import datasets, transforms
normalize = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")

transform_val = transforms.Compose([transforms.ToTensor(), normalize]) #careful to keep this one same
#We need two copies of this due to weird dataset api
cifar_data_val = datasets.CIFAR10(root='.',train=True, transform=transform_val, download=True)


## Data augmentation ##

# Baseline (no augmentation)
transform_train_0 = transforms.Compose([transforms.ToTensor(), normalize])
cifar_data_0 = datasets.CIFAR10(root='.',train=True, transform=transform_train_0, download=True)

# Cropping + Flip
transform_train_1 = transforms.Compose([
    transforms.RandomCrop(32, padding=4),    # recadrage aléatoire avec remplissage
    transforms.RandomHorizontalFlip(),       # rdm horizontal flip
    transforms.ToTensor(),
    normalize
])

transform_dict = {
    0: transform_train_0,
    1: transform_train_1,
}

In [12]:
# Training and evaluation loop across multiple seeds
accs = []
n_seeds = 50

for seed in range(1, n_seeds + 1):
    print(f"\n=== Seed {seed} ===")

    prng = RandomState(seed)
    random_permute = prng.permutation(np.arange(0, 1000))
    classes = prng.permutation(np.arange(0, 10))

    cifar_data = datasets.CIFAR10(
        root='.',
        train=True,
        transform=transform_dict[1],  # data augmentation active
        download=True
    )

    indx_train = np.concatenate([
        np.where(np.array(cifar_data.targets) == classe)[0][random_permute[0:25]]
        for classe in classes[0:2]
    ])
    indx_val = np.concatenate([
        np.where(np.array(cifar_data_val.targets) == classe)[0][random_permute[25:225]]
        for classe in classes[0:2]
    ])

    train_data = Subset(cifar_data, indx_train)
    val_data = Subset(cifar_data_val, indx_val)

    print(f'Num Samples For Training {len(train_data)} Num Samples For Val {len(val_data)}')

    train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_data, batch_size=32, shuffle=False)

    model = Net_wide_BatchNorm_Dropout(dropout_prob=0.2)
    model.to(device)

    optimizer = optim.SGD(
        model.parameters(),
        lr=0.01,
        momentum=0.8,
        weight_decay=0.001
    )

    for epoch in range(100):
        train(model, device, train_loader, optimizer, epoch, display=epoch % 5 == 0)

    acc = test(model, device, val_loader)
    accs.append(acc)

accs = np.array(accs)
print(f'\nAcc over {n_seeds} instances: %.2f +- %.2f' % (accs.mean(), accs.std()))


=== Seed 1 ===
Num Samples For Training 50 Num Samples For Val 400
Train Epoch: 0 [18/50 (50%)]	Loss: 1.913399
Train Epoch: 5 [18/50 (50%)]	Loss: 0.649791
Train Epoch: 10 [18/50 (50%)]	Loss: 0.619109
Train Epoch: 15 [18/50 (50%)]	Loss: 0.779519
Train Epoch: 20 [18/50 (50%)]	Loss: 0.444721
Train Epoch: 25 [18/50 (50%)]	Loss: 0.566986
Train Epoch: 30 [18/50 (50%)]	Loss: 0.495533
Train Epoch: 35 [18/50 (50%)]	Loss: 0.343301
Train Epoch: 40 [18/50 (50%)]	Loss: 0.272716
Train Epoch: 45 [18/50 (50%)]	Loss: 0.238719
Train Epoch: 50 [18/50 (50%)]	Loss: 0.260757
Train Epoch: 55 [18/50 (50%)]	Loss: 0.106546
Train Epoch: 60 [18/50 (50%)]	Loss: 0.149217
Train Epoch: 65 [18/50 (50%)]	Loss: 0.141497
Train Epoch: 70 [18/50 (50%)]	Loss: 0.225504
Train Epoch: 75 [18/50 (50%)]	Loss: 0.076834
Train Epoch: 80 [18/50 (50%)]	Loss: 0.141168
Train Epoch: 85 [18/50 (50%)]	Loss: 0.077662
Train Epoch: 90 [18/50 (50%)]	Loss: 0.131719
Train Epoch: 95 [18/50 (50%)]	Loss: 0.238942


/usr/local/lib/python3.11/dist-packages/torch/nn/_reduction.py:51: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))



Test set: Average loss: 1.3403, Accuracy: 253/400 (63.25%)


=== Seed 2 ===
Num Samples For Training 50 Num Samples For Val 400
Train Epoch: 0 [18/50 (50%)]	Loss: 2.274901
Train Epoch: 5 [18/50 (50%)]	Loss: 0.667732
Train Epoch: 10 [18/50 (50%)]	Loss: 0.746169
Train Epoch: 15 [18/50 (50%)]	Loss: 0.258684
Train Epoch: 20 [18/50 (50%)]	Loss: 0.227030
Train Epoch: 25 [18/50 (50%)]	Loss: 0.398711
Train Epoch: 30 [18/50 (50%)]	Loss: 0.152384
Train Epoch: 35 [18/50 (50%)]	Loss: 0.226032
Train Epoch: 40 [18/50 (50%)]	Loss: 0.153011
Train Epoch: 45 [18/50 (50%)]	Loss: 0.299588
Train Epoch: 50 [18/50 (50%)]	Loss: 0.281284
Train Epoch: 55 [18/50 (50%)]	Loss: 0.070711
Train Epoch: 60 [18/50 (50%)]	Loss: 0.053251
Train Epoch: 65 [18/50 (50%)]	Loss: 0.151959
Train Epoch: 70 [18/50 (50%)]	Loss: 0.109093
Train Epoch: 75 [18/50 (50%)]	Loss: 0.116300
Train Epoch: 80 [18/50 (50%)]	Loss: 0.064373
Train Epoch: 85 [18/50 (50%)]	Loss: 0.092857
Train Epoch: 90 [18/50 (50%)]	Loss: 0.103428
Train Epoch: 95 [1

# Challenge 2: Transfer Learning with Pre-trained Models

In Challenge 2, the use of pre-trained models was allowed. The objective remained the same: learn to classify two randomly chosen CIFAR-10 classes using only 25 training examples per class. The evaluation was done on 200 validation images per class.

For this part, I explored several ResNet architectures (ResNet-18 and ResNet-50) and performed extensive tuning using different learning rates, regularization values, and layer freezing strategies. My approach was based on fine-tuning: I froze early layers of the pre-trained backbone and trained only the classifier head and selected deeper layers.

All exploratory experiments, sweeps, and the use of Weights & Biases are detailed in the code appendix and the report. Here, I only include the final two model configurations that gave the best and most stable results across 30 different seeds, without any external logging or dependency.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Subset, DataLoader
from torchvision import datasets, transforms, models
import numpy as np
from numpy.random import RandomState

# Choose device
if torch.backends.mps.is_available():
    print("Device used is MPS - Apple")
    device = torch.device("mps")

elif torch.cuda.is_available():
    print("Device used is CUDA - Nvidia")
    device = torch.device("cuda")

else:
    print("Device used is CPU")
    device = torch.device("cpu")

# Build train and validation loaders for few-shot CIFAR-10
def build_dataset(batch_size, data_augmentation, seed):
    normalize = transforms.Normalize((0.4914, 0.4822, 0.4465),
                                     (0.247, 0.243, 0.261))
    transform_val = transforms.Compose([transforms.ToTensor(), normalize])
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize
    ])
    transform_dict = {
        0: transforms.Compose([transforms.ToTensor(), normalize]),
        1: transform_train
    }
    cifar_train_full = datasets.CIFAR10(root='.', train=True,
                                        transform=transform_dict[data_augmentation],
                                        download=True)
    cifar_val_full   = datasets.CIFAR10(root='.', train=True,
                                        transform=transform_val,
                                        download=True)
    prng = RandomState(seed)
    perm = prng.permutation(1000)
    classes = prng.permutation(10)
    # select 2 random classes
    train_idx = np.concatenate([
        np.where(np.array(cifar_train_full.targets) == c)[0][perm[:25]]
        for c in classes[:2]
    ])
    val_idx = np.concatenate([
        np.where(np.array(cifar_val_full.targets) == c)[0][perm[25:225]]
        for c in classes[:2]
    ])
    train_loader = DataLoader(Subset(cifar_train_full, train_idx),
                              batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(Subset(cifar_val_full,   val_idx),
                              batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

# Build ResNet18 with dynamic freezing
def build_network(model_name, freeze_layers):
    # 1) Choice of backbone
    if model_name == "resnet18":
        model = models.resnet18(pretrained=True)
    elif model_name == "resnet34":
        model = models.resnet34(pretrained=True)
    elif model_name == "resnet50":
        model = models.resnet50(pretrained=True)
    elif model_name == "resnet101":
        model = models.resnet101(pretrained=True)
    else:
        raise ValueError(f"Not known model {model_name}")

    # always freeze conv1 & bn1
    for p in model.conv1.parameters(): p.requires_grad = False
    for p in model.bn1.parameters():  p.requires_grad = False
    # freeze first N residual blocks
    blocks = [model.layer1, model.layer2, model.layer3, model.layer4]
    for idx, block in enumerate(blocks, start=1):
        freeze = idx <= freeze_layers
        for p in block.parameters(): p.requires_grad = not freeze
    # always unfreeze classifier
    model.fc = nn.Linear(model.fc.in_features, 10)
    for p in model.fc.parameters(): p.requires_grad = True
    return model.to(device)

# Build SGD optimizer
def build_optimizer(model, lr, momentum, weight_decay):
    params = filter(lambda p: p.requires_grad, model.parameters())
    return optim.SGD(params, lr=lr, momentum=momentum, weight_decay=weight_decay)

# Single epoch training
def train_epoch(model, device, loader, optimizer, epoch, display=False):
    model.train()
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        if display and batch_idx == 0:
            print(f'Train Epoch: {epoch} [{len(data)}/{len(loader.dataset)} (50%)]\tLoss: {loss.item():.6f}')
    return loss.item()

# Evaluation on validation set
def evaluate(model, device, loader):
    model.eval()
    test_loss = 0.0
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.cross_entropy(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(loader.dataset)
    acc = 100. * correct / len(loader.dataset)
    print(f"\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(loader.dataset)} ({acc:.2f}%)\n")
    return test_loss, acc

Device used is CUDA - Nvidia


In [2]:
# Run over multiple seeds
accs = []
n_seeds = 30
for seed in range(1, n_seeds + 1):
    print(f"=== Seed {seed} ===")

    train_loader, val_loader = build_dataset(batch_size=32, data_augmentation=1, seed=seed)
    model = build_network(model_name="resnet50", freeze_layers=1)
    optimizer = build_optimizer(model, lr=0.001, momentum=0.9, weight_decay=0.0005)

    for epoch in range(100):
        train_epoch(model, device, train_loader, optimizer, epoch, display=(epoch % 5 == 0))

    _, acc = evaluate(model, device, val_loader)
    accs.append(acc)

accs = np.array(accs)
print(f'Acc over {n_seeds} instances: {accs.mean():.2f} +- {accs.std():.2f}')

=== Seed 1 ===


100%|██████████| 170M/170M [00:03<00:00, 43.3MB/s]
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 155MB/s]


Train Epoch: 0 [32/50 (50%)]	Loss: 2.217705
Train Epoch: 5 [32/50 (50%)]	Loss: 0.626853
Train Epoch: 10 [32/50 (50%)]	Loss: 0.401700
Train Epoch: 15 [32/50 (50%)]	Loss: 0.207628
Train Epoch: 20 [32/50 (50%)]	Loss: 0.087201
Train Epoch: 25 [32/50 (50%)]	Loss: 0.063699
Train Epoch: 30 [32/50 (50%)]	Loss: 0.044259
Train Epoch: 35 [32/50 (50%)]	Loss: 0.015284
Train Epoch: 40 [32/50 (50%)]	Loss: 0.035814
Train Epoch: 45 [32/50 (50%)]	Loss: 0.082651
Train Epoch: 50 [32/50 (50%)]	Loss: 0.060663
Train Epoch: 55 [32/50 (50%)]	Loss: 0.041683
Train Epoch: 60 [32/50 (50%)]	Loss: 0.006802
Train Epoch: 65 [32/50 (50%)]	Loss: 0.060781
Train Epoch: 70 [32/50 (50%)]	Loss: 0.007147
Train Epoch: 75 [32/50 (50%)]	Loss: 0.009081
Train Epoch: 80 [32/50 (50%)]	Loss: 0.007796
Train Epoch: 85 [32/50 (50%)]	Loss: 0.006174
Train Epoch: 90 [32/50 (50%)]	Loss: 0.021405
Train Epoch: 95 [32/50 (50%)]	Loss: 0.089765

Test set: Average loss: 0.7793, Accuracy: 284/400 (71.00%)

=== Seed 2 ===
Train Epoch: 0 [32/50 (50%